# Direct questioning — DeepSeek-V3 on COM2SENSE (500 claims, sample hash 3d1fbba18f)

The model is asked whether the statement is true or false (temperature 0, no facts). Result: 73.4. Run on Kaggle; the API key is read from Kaggle secrets.

In [3]:
# ===== DIRECT QUESTIONING — DeepSeek + COM2SENSE (same 500-claim sample, no facts) =====
!pip install -q openai scikit-learn

import os, json, time, pickle, hashlib, base64, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from IPython.display import HTML, display

MODEL_NAME = "deepseek-chat"
LABEL      = "DeepSeek-V3"
DATASET    = "COM2SENSE"
RUN_NAME   = "direct_deepseek_com2sense"
client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"),
                base_url="https://api.deepseek.com", timeout=180)

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
DCK = WORK_DIR / f"{RUN_NAME}_ckpt.pkl"

# ---- SAME COM2SENSE sample (seed 42) ----
C2S_REPO = "/kaggle/working/Com2Sense"
if not Path(C2S_REPO).exists():
    os.system(f"git clone -q --depth 1 https://github.com/PlusLabNLP/Com2Sense.git {C2S_REPO}")
raw = pd.DataFrame(json.load(open(f"{C2S_REPO}/data/train.json")))
raw = raw[raw["label"].isin(["True", "False"])].copy()
raw["gold"] = (raw["label"] == "True").astype(int)
raw = raw.sort_values("id").reset_index(drop=True)
SEED = 42
true_part  = raw[raw["gold"] == 1].sample(n=250, random_state=SEED)
false_part = raw[raw["gold"] == 0].sample(n=250, random_state=SEED)
sample = pd.concat([true_part, false_part]).sample(frac=1, random_state=SEED).reset_index(drop=True)
claims = sample["sent"].astype(str).str.strip().tolist()
gold   = sample["gold"].astype(int).tolist()
SAMPLE_HASH = hashlib.md5("|".join(sample["id"]).encode()).hexdigest()[:10]
print("claims:", len(claims), "| true:", sum(gold), "| SAMPLE HASH:", SAMPLE_HASH)
assert SAMPLE_HASH == "3d1fbba18f", "Different sample! It must be the same 500 claims as our method."


def call(prompt, temperature, max_tokens):
    last = None
    for a in range(1, 9):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, temperature=temperature, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}])
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e
            if "402" in str(e):
                raise RuntimeError("no API credit left")
            busy = any(c in str(e) for c in ["429", "502", "503", "504"])
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def parse_tf(text):
    t = (text or "").strip().lower().lstrip("*\"' ")
    if t.startswith("true"):
        return 1
    if t.startswith("false"):
        return 0
    has_t, has_f = "true" in t, "false" in t
    if has_t and not has_f:
        return 1
    if has_f and not has_t:
        return 0
    return None


def ask(claim):
    p = ('Is the following statement true or false? Answer with only '
         f'one word: "true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    try:
        return parse_tf(call(p, 0, 10))
    except Exception:
        return None


dres = pickle.load(open(DCK, "rb")) if DCK.exists() else {}
print("already done:", len(dres))
for i in range(len(claims)):
    if i in dres:
        continue
    dres[i] = ask(claims[i])
    pickle.dump(dres, open(DCK, "wb"))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(claims)} | unparsed so far: {sum(v is None for v in dres.values())}", flush=True)

pred = [dres[i] if dres[i] is not None else 0 for i in range(len(claims))]
fails = sum(dres[i] is None for i in range(len(claims)))
tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
res = pd.DataFrame([{"method": "Direct (no facts)", "n": len(pred),
                     "acc": round(accuracy_score(gold, pred) * 100, 1),
                     "prec": round(precision_score(gold, pred, zero_division=0) * 100, 1),
                     "rec": round(recall_score(gold, pred, zero_division=0) * 100, 1),
                     "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp, "unparsed": fails}])
pd.set_option("display.width", 220)
print(f"\n=== FINAL: {LABEL} | {DATASET} | Direct | sample {SAMPLE_HASH} ===")
print(res.to_string(index=False))

# ---- backup: embedded Download button ----
csv = WORK_DIR / f"{RUN_NAME}_results.csv"
res.to_csv(csv, index=False)
pkl = WORK_DIR / f"{RUN_NAME}_{len(dres)}.pkl"
pickle.dump({"claims": claims, "gold": gold, "pred": dres, "sample_hash": SAMPLE_HASH}, open(pkl, "wb"))
zp = WORK_DIR / f"{RUN_NAME}_backup.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pkl, arcname=pkl.name)
    z.write(csv, arcname=csv.name)
b64 = base64.b64encode(open(zp, "rb").read()).decode()
display(HTML(
    f'<a download="{zp.name}" href="data:application/zip;base64,{b64}" '
    f'style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;'
    f'border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download {zp.name}</a>'))

claims: 500 | true: 250 | SAMPLE HASH: 3d1fbba18f
already done: 0
  50/500 | unparsed so far: 0
  100/500 | unparsed so far: 0
  150/500 | unparsed so far: 0
  200/500 | unparsed so far: 0
  250/500 | unparsed so far: 0
  300/500 | unparsed so far: 0
  350/500 | unparsed so far: 0
  400/500 | unparsed so far: 0
  450/500 | unparsed so far: 0
  500/500 | unparsed so far: 0

=== FINAL: DeepSeek-V3 | COM2SENSE | Direct | sample 3d1fbba18f ===
           method   n  acc  prec  rec   f1  TN  FP  FN  TP  unparsed
Direct (no facts) 500 73.4  89.8 52.8 66.5 235  15 118 132         0
